# YOLO11 Ensemble Inference

Notebook này sử dụng phương pháp ensemble để kết hợp kết quả từ nhiều phiên bản YOLO khác nhau nhằm cải thiện độ chính xác detection.


In [ ]:
import os
import json
import cv2
import numpy as np
from tqdm import tqdm
from ultralytics import YOLO
import torch

# Thư viện để kết hợp bounding boxes từ nhiều model
try:
    from ensemble_boxes import weighted_boxes_fusion
    WBF_AVAILABLE = True
except ImportError:
    print("Warning: ensemble-boxes not installed. Installing...")
    import subprocess
    subprocess.check_call(["pip", "install", "ensemble-boxes"])
    from ensemble_boxes import weighted_boxes_fusion
    WBF_AVAILABLE = True


## Configuration


In [ ]:
# Đường dẫn đến các model đã train
# Có thể sử dụng các phiên bản khác nhau: yolo11s, yolo11m, yolo11l, yolo11x
MODEL_PATHS = [
    'runs/detect/drone_training_yolo11s/weights/best.pt',
    # Thêm các model khác nếu có:
    # 'runs/detect/drone_training_yolo11m/weights/best.pt',
    # 'runs/detect/drone_training_yolo11l/weights/best.pt',
    # 'runs/detect/drone_training_yolo11x/weights/best.pt',
]

# Trọng số cho mỗi model (có thể điều chỉnh dựa trên performance)
# Model tốt hơn nên có trọng số cao hơn
MODEL_WEIGHTS = [1.0, 1.0, 1.0, 1.0]  # Điều chỉnh theo số lượng model

# Cấu hình inference
TEST_DATA_DIR = 'public_test/samples/'
OUTPUT_FILE = 'predictions_ensemble.json'
CONFIDENCE_THRESHOLD = 0.25  # Threshold ban đầu cho mỗi model

# Cấu hình Weighted Boxes Fusion
WBF_IOU_THRESH = 0.5  # IoU threshold cho WBF
WBF_SKIP_BOX_THRESH = 0.0001  # Skip boxes với confidence quá thấp

# Device
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"Using device: {DEVICE}")


## Load Models


In [ ]:
def load_ensemble_models(model_paths):
    """
    Load tất cả các model YOLO cho ensemble.
    
    Args:
        model_paths: List các đường dẫn đến model weights
    
    Returns:
        List các model đã load
    """
    models = []
    
    for i, model_path in enumerate(model_paths):
        if not os.path.exists(model_path):
            print(f"Warning: Model {i+1} not found at {model_path}, skipping...")
            continue
            
        try:
            model = YOLO(model_path)
            model.to(DEVICE)
            models.append(model)
            print(f"✓ Loaded model {i+1}: {os.path.basename(model_path)}")
        except Exception as e:
            print(f"Error loading model {model_path}: {e}")
            continue
    
    if len(models) == 0:
        raise ValueError("No models were successfully loaded!")
    
    print(f"\nSuccessfully loaded {len(models)} model(s) for ensemble.")
    return models

# Load tất cả các model
ensemble_models = load_ensemble_models(MODEL_PATHS)

# Điều chỉnh weights nếu số lượng model khác với số weights
if len(MODEL_WEIGHTS) != len(ensemble_models):
    MODEL_WEIGHTS = [1.0] * len(ensemble_models)
    print(f"Adjusted model weights to: {MODEL_WEIGHTS}")


## Ensemble Functions


In [ ]:
def run_ensemble_inference_on_frame(models, frame, conf_threshold=0.25):
    """
    Chạy inference với tất cả các model trên một frame và thu thập kết quả.
    
    Args:
        models: List các YOLO models
        frame: Frame image (numpy array)
        conf_threshold: Confidence threshold
    
    Returns:
        List các detection boxes từ tất cả models
    """
    all_boxes = []
    all_scores = []
    all_labels = []
    
    img_h, img_w = frame.shape[:2]
    
    for model in models:
        # Chạy inference
        results = model.predict(
            frame,
            conf=conf_threshold,
            verbose=False,
            device=DEVICE
        )
        
        result = results[0]
        
        # Lấy boxes, scores, labels
        if result.boxes is not None and len(result.boxes) > 0:
            boxes = result.boxes.xyxy.cpu().numpy()  # [x1, y1, x2, y2]
            scores = result.boxes.conf.cpu().numpy()  # confidence scores
            labels = result.boxes.cls.cpu().numpy().astype(int)  # class labels
            
            # Normalize boxes to [0, 1] range (required by WBF)
            normalized_boxes = boxes.copy()
            normalized_boxes[:, [0, 2]] /= img_w  # normalize x coordinates
            normalized_boxes[:, [1, 3]] /= img_h  # normalize y coordinates
            
            all_boxes.append(normalized_boxes)
            all_scores.append(scores)
            all_labels.append(labels)
        else:
            # Không có detection nào
            all_boxes.append(np.array([]).reshape(0, 4))
            all_scores.append(np.array([]))
            all_labels.append(np.array([]))
    
    return all_boxes, all_scores, all_labels


def fuse_detections_with_wbf(all_boxes, all_scores, all_labels, img_w, img_h, 
                             weights=None, iou_thr=0.5, skip_box_thr=0.0001):
    """
    Kết hợp detections từ nhiều model sử dụng Weighted Boxes Fusion.
    
    Args:
        all_boxes: List các numpy arrays chứa normalized boxes [x1, y1, x2, y2]
        all_scores: List các numpy arrays chứa confidence scores
        all_labels: List các numpy arrays chứa class labels
        img_w: Image width
        img_h: Image height
        weights: List weights cho mỗi model
        iou_thr: IoU threshold cho WBF
        skip_box_thr: Skip boxes với confidence quá thấp
    
    Returns:
        Fused boxes, scores, labels (denormalized)
    """
    if weights is None:
        weights = [1.0] * len(all_boxes)
    
    # WBF yêu cầu format đặc biệt
    boxes_list = []
    scores_list = []
    labels_list = []
    
    for boxes, scores, labels in zip(all_boxes, all_scores, all_labels):
        if len(boxes) > 0:
            boxes_list.append(boxes.tolist())
            scores_list.append(scores.tolist())
            labels_list.append(labels.tolist())
        else:
            boxes_list.append([])
            scores_list.append([])
            labels_list.append([])
    
    # Chạy Weighted Boxes Fusion
    try:
        boxes, scores, labels = weighted_boxes_fusion(
            boxes_list,
            scores_list,
            labels_list,
            weights=weights,
            iou_thr=iou_thr,
            skip_box_thr=skip_box_thr
        )
        
        # Denormalize boxes
        if len(boxes) > 0:
            boxes = np.array(boxes)
            boxes[:, [0, 2]] *= img_w
            boxes[:, [1, 3]] *= img_h
            
            return boxes, np.array(scores), np.array(labels).astype(int)
        else:
            return np.array([]).reshape(0, 4), np.array([]), np.array([])
            
    except Exception as e:
        print(f"Error in WBF: {e}")
        return np.array([]).reshape(0, 4), np.array([]), np.array([])


## Main Inference Function


In [ ]:
def run_ensemble_inference():
    """
    Chạy ensemble inference trên tất cả test videos.
    """
    
    # Tìm tất cả video folders
    try:
        video_folders = sorted([f for f in os.listdir(TEST_DATA_DIR) 
                               if os.path.isdir(os.path.join(TEST_DATA_DIR, f))])
    except FileNotFoundError:
        print(f"Error: Test data directory not found at: {TEST_DATA_DIR}")
        return
        
    if not video_folders:
        print(f"Error: No video folders found in {TEST_DATA_DIR}")
        return

    print(f"Found {len(video_folders)} videos to process...")
    print(f"Using {len(ensemble_models)} model(s) for ensemble\n")

    all_predictions = []

    # Process từng video
    for video_folder_name in tqdm(video_folders, desc="Processing videos"):
        video_path = os.path.join(TEST_DATA_DIR, video_folder_name, 'drone_video.mp4')
        
        if not os.path.exists(video_path):
            print(f"Warning: 'drone_video.mp4' not found in {video_folder_name}, skipping.")
            continue
            
        video_bboxes = []

        try:
            # Mở video
            cap = cv2.VideoCapture(video_path)
            if not cap.isOpened():
                print(f"Error: Could not open video {video_path}")
                continue
            
            frame_idx = 0
            
            while True:
                ret, frame = cap.read()
                if not ret:
                    break
                
                # Chạy ensemble inference trên frame này
                all_boxes, all_scores, all_labels = run_ensemble_inference_on_frame(
                    ensemble_models,
                    frame,
                    conf_threshold=CONFIDENCE_THRESHOLD
                )
                
                # Kết hợp kết quả bằng WBF
                img_h, img_w = frame.shape[:2]
                fused_boxes, fused_scores, fused_labels = fuse_detections_with_wbf(
                    all_boxes,
                    all_scores,
                    all_labels,
                    img_w,
                    img_h,
                    weights=MODEL_WEIGHTS[:len(ensemble_models)],
                    iou_thr=WBF_IOU_THRESH,
                    skip_box_thr=WBF_SKIP_BOX_THRESH
                )
                
                # Lưu các detections
                if len(fused_boxes) > 0:
                    for box, score in zip(fused_boxes, fused_scores):
                        x1, y1, x2, y2 = box
                        
                        bbox_data = {
                            "frame": frame_idx,
                            "x1": int(round(x1)),
                            "y1": int(round(y1)),
                            "x2": int(round(x2)),
                            "y2": int(round(y2))
                        }
                        video_bboxes.append(bbox_data)
                
                frame_idx += 1
            
            cap.release()
        
        except Exception as e:
            print(f"Error while processing video {video_path}: {e}")
            continue

        # Tạo JSON structure
        detections_list = []
        if len(video_bboxes) > 0:
            detections_list.append({"bboxes": video_bboxes})

        final_video_obj = {
            "video_id": video_folder_name,
            "detections": detections_list
        }
        all_predictions.append(final_video_obj)

    # Lưu kết quả
    try:
        print(f"\nSaving all {len(all_predictions)} video predictions to {OUTPUT_FILE}...")
        with open(OUTPUT_FILE, 'w') as f:
            json.dump(all_predictions, f, indent=4)
        print("Ensemble inference complete!")
    except Exception as e:
        print(f"Error: Could not write output JSON file: {e}")

# Chạy inference
run_ensemble_inference()


## Alternative: NMS-based Ensemble (Optional)


In [ ]:
def fuse_detections_with_nms(all_boxes, all_scores, all_labels, img_w, img_h, 
                             iou_threshold=0.5, conf_threshold=0.25):
    """
    Kết hợp detections bằng cách gộp tất cả boxes lại và áp dụng NMS.
    Đây là phương pháp đơn giản hơn WBF.
    """
    # Gộp tất cả boxes từ các model
    all_boxes_combined = []
    all_scores_combined = []
    all_labels_combined = []
    
    for boxes, scores, labels in zip(all_boxes, all_scores, all_labels):
        if len(boxes) > 0:
            # Denormalize
            denorm_boxes = boxes.copy()
            denorm_boxes[:, [0, 2]] *= img_w
            denorm_boxes[:, [1, 3]] *= img_h
            
            all_boxes_combined.append(denorm_boxes)
            all_scores_combined.append(scores)
            all_labels_combined.append(labels)
    
    if len(all_boxes_combined) == 0:
        return np.array([]).reshape(0, 4), np.array([]), np.array([])
    
    # Concatenate tất cả
    boxes_combined = np.vstack(all_boxes_combined)
    scores_combined = np.concatenate(all_scores_combined)
    labels_combined = np.concatenate(all_labels_combined)
    
    # Áp dụng NMS
    indices = cv2.dnn.NMSBoxes(
        boxes_combined.tolist(),
        scores_combined.tolist(),
        conf_threshold,
        iou_threshold
    )
    
    if len(indices) > 0:
        indices = indices.flatten()
        return boxes_combined[indices], scores_combined[indices], labels_combined[indices]
    else:
        return np.array([]).reshape(0, 4), np.array([]), np.array([])


# Có thể sử dụng hàm này thay cho WBF nếu muốn:
# fused_boxes, fused_scores, fused_labels = fuse_detections_with_nms(
#     all_boxes, all_scores, all_labels, img_w, img_h
# )


## Visualization (Optional)


In [ ]:
def visualize_ensemble_results(video_path, models, frame_idx=0):
    """
    Visualize kết quả ensemble trên một frame cụ thể.
    """
    import matplotlib.pyplot as plt
    
    cap = cv2.VideoCapture(video_path)
    cap.set(cv2.CAP_PROP_POS_FRAMES, frame_idx)
    ret, frame = cap.read()
    cap.release()
    
    if not ret:
        print(f"Could not read frame {frame_idx}")
        return
    
    # Chạy inference
    all_boxes, all_scores, all_labels = run_ensemble_inference_on_frame(
        models, frame, CONFIDENCE_THRESHOLD
    )
    
    # Fuse
    img_h, img_w = frame.shape[:2]
    fused_boxes, fused_scores, fused_labels = fuse_detections_with_wbf(
        all_boxes, all_scores, all_labels, img_w, img_h,
        weights=MODEL_WEIGHTS[:len(models)],
        iou_thr=WBF_IOU_THRESH
    )
    
    # Vẽ boxes
    frame_vis = frame.copy()
    for box, score in zip(fused_boxes, fused_scores):
        x1, y1, x2, y2 = box.astype(int)
        cv2.rectangle(frame_vis, (x1, y1), (x2, y2), (0, 255, 0), 2)
        cv2.putText(frame_vis, f"{score:.2f}", (x1, y1-10),
                   cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 255, 0), 2)
    
    # Hiển thị
    frame_rgb = cv2.cvtColor(frame_vis, cv2.COLOR_BGR2RGB)
    plt.figure(figsize=(12, 8))
    plt.imshow(frame_rgb)
    plt.axis('off')
    plt.title(f"Ensemble Results - Frame {frame_idx} ({len(fused_boxes)} detections)")
    plt.show()


# Ví dụ sử dụng:
# test_video = 'public_test/samples/drone_video_001/drone_video.mp4'
# if os.path.exists(test_video):
#     visualize_ensemble_results(test_video, ensemble_models, frame_idx=100)
